# 02 — Trading Activity

**Investment question:** How was the portfolio traded, what did trading cost, and what realized results did it produce?

This notebook reads the private `analytics_trade_orders` table. `trade_side` is derived from signed quantity. Outputs must be cleared before committing.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine

sns.set_theme(style="whitegrid")
load_dotenv()
engine = create_engine(os.environ["DATABASE_URL"])

## Load and validate curated trading records

In [ ]:
query = """
SELECT "Asset_category" AS asset_category, currency, symbol, executed_at, quantity, trade_price,
       close_price, proceeds, commission_fee, basis, realized_pnl, mtm_pnl, code
FROM analytics_trade_orders
ORDER BY executed_at
"""
trades = pd.read_sql(query, engine, parse_dates=["executed_at"])
trades["trade_side"] = np.where(trades["quantity"] > 0, "BUY", "SELL")
assert trades["symbol"].notna().all()
assert trades["executed_at"].notna().all()
assert (trades["quantity"] != 0).all()
assert (trades["trade_price"] > 0).all()
assert (((trades["quantity"] > 0) & (trades["proceeds"] < 0)) | ((trades["quantity"] < 0) & (trades["proceeds"] > 0))).all()
print(f"Records: {len(trades):,}")
print(f"Symbols traded: {trades['symbol'].nunique():,}")
print(f"Period: {trades['executed_at'].min()} to {trades['executed_at'].max()}")

## Buy versus sell activity

In [ ]:
side_summary = (
    trades.groupby("trade_side")
    .agg(
        trade_records=("symbol", "size"),
        unique_symbols=("symbol", "nunique"),
        gross_traded_amount=("proceeds", lambda x: x.abs().sum()),
        average_trade_amount=("proceeds", lambda x: x.abs().mean()),
    )
)
side_summary["record_pct"] = side_summary["trade_records"] / len(trades) * 100
side_summary.round(2)

## Monthly activity

In [ ]:
trades["month"] = trades["executed_at"].dt.to_period("M").dt.to_timestamp()
monthly_activity = (
    trades.groupby("month")
    .agg(
        total_records=("symbol", "size"),
        buy_count=("trade_side", lambda x: x.eq("BUY").sum()),
        sell_count=("trade_side", lambda x: x.eq("SELL").sum()),
        gross_traded_amount=("proceeds", lambda x: x.abs().sum()),
        net_trading_cash_flow=("proceeds", "sum"),
    )
)
monthly_activity.round(2)

In [ ]:
ax = monthly_activity["gross_traded_amount"].plot(kind="bar", figsize=(10, 5), color="#2563eb")
ax.set(title="Monthly gross trading activity", xlabel="Month", ylabel="Gross traded amount")
ax.set_xticklabels([d.strftime("%Y-%m") for d in monthly_activity.index], rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Activity by symbol and concentration

In [ ]:
symbol_activity = (
    trades.groupby("symbol")
    .agg(
        trade_records=("symbol", "size"),
        buy_count=("trade_side", lambda x: x.eq("BUY").sum()),
        sell_count=("trade_side", lambda x: x.eq("SELL").sum()),
        gross_traded_amount=("proceeds", lambda x: x.abs().sum()),
        net_trading_cash_flow=("proceeds", "sum"),
    )
    .sort_values("gross_traded_amount", ascending=False)
)
volume_total = symbol_activity["gross_traded_amount"].sum()
frequency_total = symbol_activity["trade_records"].sum()
concentration = pd.Series({
    "traded_symbols": len(symbol_activity),
    "top_5_volume_pct": symbol_activity.head(5)["gross_traded_amount"].sum() / volume_total * 100,
    "top_10_volume_pct": symbol_activity.head(10)["gross_traded_amount"].sum() / volume_total * 100,
    "top_5_frequency_pct": symbol_activity.nlargest(5, "trade_records")["trade_records"].sum() / frequency_total * 100,
    "top_10_frequency_pct": symbol_activity.nlargest(10, "trade_records")["trade_records"].sum() / frequency_total * 100,
})
display(symbol_activity.head(15).round(2))
display(concentration.round(2))

## Commissions and IBKR P&L reconciliation

In [ ]:
commission_metrics = pd.Series({
    "negative_commissions": trades["commission_fee"].lt(0).sum(),
    "zero_commissions": trades["commission_fee"].eq(0).sum(),
    "total_commission_cost": trades["commission_fee"].abs().sum(),
    "average_commission": trades["commission_fee"].abs().mean(),
    "largest_commission": trades["commission_fee"].abs().max(),
})
commission_metrics.round(4)

In [ ]:
sell_records = trades.loc[trades["quantity"] < 0].copy()
sell_records["calculated_pnl"] = sell_records["proceeds"] + sell_records["commission_fee"] + sell_records["basis"]
sell_records["pnl_difference"] = sell_records["realized_pnl"] - sell_records["calculated_pnl"]
exceptions = sell_records.loc[sell_records["pnl_difference"].abs() > 0.01, ["executed_at", "symbol", "realized_pnl", "calculated_pnl", "pnl_difference", "code"]]
print(f"Ordinary formula matches: {(sell_records['pnl_difference'].abs() <= 0.01).sum():,}")
print(f"Mixed/exception records: {len(exceptions):,}")
exceptions

Rows coded `C;O` combine closing and opening activity. Reported IBKR `realized_pnl` is authoritative and must not have commissions subtracted again.

## Definitive realized P&L metrics

In [ ]:
realized = trades.loc[trades["realized_pnl"] != 0].copy()
gross_profit = realized.loc[realized["realized_pnl"] > 0, "realized_pnl"].sum()
gross_loss = realized.loc[realized["realized_pnl"] < 0, "realized_pnl"].abs().sum()
pnl_metrics = pd.Series({
    "records_with_realized_pnl": len(realized),
    "profitable_records": realized["realized_pnl"].gt(0).sum(),
    "losing_records": realized["realized_pnl"].lt(0).sum(),
    "net_realized_pnl": realized["realized_pnl"].sum(),
    "gross_realized_profit": gross_profit,
    "gross_realized_loss": gross_loss,
    "profitable_records_pct": realized["realized_pnl"].gt(0).mean() * 100,
    "profit_factor": gross_profit / gross_loss,
})
pnl_metrics.round(2)

## Monthly P&L and contributors

In [ ]:
monthly_pnl = (
    trades.groupby("month")
    .agg(
        records_with_realized_pnl=("realized_pnl", lambda x: x.ne(0).sum()),
        profitable_records=("realized_pnl", lambda x: x.gt(0).sum()),
        losing_records=("realized_pnl", lambda x: x.lt(0).sum()),
        net_realized_pnl=("realized_pnl", "sum"),
        gross_realized_profit=("realized_pnl", lambda x: x[x > 0].sum()),
        gross_realized_loss=("realized_pnl", lambda x: x[x < 0].abs().sum()),
    )
)
monthly_pnl.round(2)

In [ ]:
colors = np.where(monthly_pnl["net_realized_pnl"] >= 0, "#16a34a", "#dc2626")
ax = monthly_pnl["net_realized_pnl"].plot(kind="bar", figsize=(10, 5), color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Monthly realized P&L", xlabel="Month", ylabel="Realized P&L")
ax.set_xticklabels([d.strftime("%Y-%m") for d in monthly_pnl.index], rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
symbol_pnl = (
    trades.groupby("symbol")
    .agg(
        records_with_realized_pnl=("realized_pnl", lambda x: x.ne(0).sum()),
        profitable_records=("realized_pnl", lambda x: x.gt(0).sum()),
        losing_records=("realized_pnl", lambda x: x.lt(0).sum()),
        net_realized_pnl=("realized_pnl", "sum"),
    )
)
top_gainers = symbol_pnl.nlargest(10, "net_realized_pnl")
top_detractors = symbol_pnl.nsmallest(10, "net_realized_pnl")
display(top_gainers.round(2))
display(top_detractors.round(2))

In [ ]:
contributors = pd.concat([top_gainers, top_detractors]).sort_values("net_realized_pnl")
colors = np.where(contributors["net_realized_pnl"] >= 0, "#16a34a", "#dc2626")
ax = contributors["net_realized_pnl"].plot(kind="barh", figsize=(9, 8), color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title="Leading realized P&L contributors and detractors", xlabel="Net realized P&L", ylabel="Symbol")
plt.tight_layout()
plt.show()

## Interpretation checklist

- Compare frequency with monetary trading volume.
- Identify months dominated by capital deployment or liquidation.
- Evaluate whether activity is concentrated in a small group of symbols.
- Separate commission costs from P&L without double-counting them.
- Treat reported realized P&L as authoritative for mixed closing/opening executions.
- Do not interpret realized P&L as total portfolio return; open positions and external cash flows are excluded.